<a href="https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/risk/lopa_sis_risk_framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LOPA and safety-instrumented risk analysis with NeqSim

This notebook demonstrates the safety-layer part of the NeqSim risk framework using the current `SISIntegratedRiskModel`, `SafetyInstrumentedFunction`, and `LOPAResult` APIs.

The numerical values are deliberately transparent and illustrative. They demonstrate software behavior, not approval of initiating-event frequencies, IPL independence, SIL targets, proof-test intervals, or safe-state definitions. Those inputs require a traceable hazard study and accountable engineering review.

Documentation: https://equinor.github.io/neqsim/risk/index.html

In [ ]:
%pip -q install neqsim

from neqsim import jneqsim
import pandas as pd
import matplotlib.pyplot as plt

RiskEvent = jneqsim.process.safety.risk.RiskEvent
SISIntegratedRiskModel = jneqsim.process.safety.risk.sis.SISIntegratedRiskModel
SafetyInstrumentedFunction = jneqsim.process.safety.risk.sis.SafetyInstrumentedFunction

print('NeqSim SIS/LOPA classes loaded')

## 1. Define an initiating event

We use a high-pressure vessel overpressure event with an illustrative initiating frequency of 0.1 per year. The model stores the event name, frequency, and consequence category.

In [ ]:
event_name = 'HP vessel overpressure'
f_ie = 0.1

model = SISIntegratedRiskModel('HP vessel LOPA')
model.addInitiatingEvent(event_name, f_ie, RiskEvent.ConsequenceCategory.MAJOR)
print(f'Initiating event frequency: {f_ie:.3g} per year')

## 2. Add an independent protection layer

Here a BPCS pressure-control layer is assigned PFD = 0.1. In a real LOPA, independence, auditability, response time, common-cause effects, and the credited PFD must be justified before the layer is accepted as an IPL.

In [ ]:
bpcs_pfd = 0.1
bpcs = SISIntegratedRiskModel.IndependentProtectionLayer(
    'BPCS pressure control',
    bpcs_pfd,
    SISIntegratedRiskModel.IndependentProtectionLayer.IPLType.BPCS,
)
bpcs.addApplicableEvent(event_name)
model.addIPL(bpcs)
print(f'BPCS PFD: {bpcs_pfd:.3g}')

## 3. Define a safety instrumented function

The NeqSim builder API gives the SIF a traceable identifier, description, SIL label, PFD, protected equipment, and safe state. The example uses PFD = 0.005.

In [ ]:
sif_pfd = 0.005
esd = (SafetyInstrumentedFunction.builder()
       .id('SIF-001')
       .name('High-pressure ESD')
       .description('Isolate the HP vessel on confirmed high pressure')
       .sil(2)
       .pfd(sif_pfd)
       .initiatingEvent(event_name)
       .addProtectedEquipment('HP vessel')
       .safeState('Isolated')
       .build())
model.addSIF(esd)
print('SIF-001 added with target label SIL 2 and PFD =', sif_pfd)

## 4. Perform the LOPA and independently close the arithmetic

For independent layers the simple frequency multiplication in this example is

`f_mitigated = f_IE × PFD_BPCS × PFD_SIF`.

The independent Python calculation below is included as a notebook validation gate.

In [ ]:
result = model.performLOPA(event_name)
expected = f_ie * bpcs_pfd * sif_pfd
actual = result.getMitigatedFrequency()

print(f'Expected mitigated frequency: {expected:.3e} per year')
print(f'NeqSim mitigated frequency:   {actual:.3e} per year')
print(f'Total risk-reduction factor:  {result.getTotalRRF():.1f}')

assert abs(actual - expected) < 1.0e-12
assert abs(result.getTotalRRF() - 1.0/(bpcs_pfd*sif_pfd)) < 1.0e-8

## 5. Sensitivity: which credited layer dominates residual frequency?

A simple sensitivity sweep helps explain the multiplicative nature of LOPA and is useful for teaching. It must not be interpreted as permission to tune PFD values without engineering justification.

In [ ]:
sif_pfds = [0.1, 0.03, 0.01, 0.005, 0.003, 0.001]
rows = []
for pfd in sif_pfds:
    rows.append({
        'SIF PFD': pfd,
        'mitigated frequency [/yr]': f_ie * bpcs_pfd * pfd,
        'total RRF': 1.0/(bpcs_pfd*pfd),
    })
sens = pd.DataFrame(rows)
display(sens)

plt.figure(figsize=(7,4))
plt.loglog(sens['SIF PFD'], sens['mitigated frequency [/yr]'], marker='o')
plt.xlabel('SIF PFD')
plt.ylabel('Mitigated frequency [1/yr]')
plt.title('LOPA sensitivity to SIF PFD')
plt.grid(True, which='both')
plt.tight_layout()
plt.show()

## 6. Connect safety risk to process simulation

The next step in a full NeqSim workflow is to replace a generic consequence category with process-derived evidence: vessel inventory, pressure and temperature, relief/depressurization response, production impact, dynamic shutdown behavior, and downstream dependencies. NeqSim's risk documentation also covers Monte Carlo simulation, dynamic risk, bow-tie analysis, process topology, condition-based reliability, and real-time risk monitoring.

A useful agentic workflow is: **process state → hazardous scenario → physics calculation → protection-layer model → residual risk result → traceable engineering recommendation**. The agent coordinates the work; the hazard-study assumptions and acceptance decision remain governed engineering inputs.